# Manzanas en video
YOLO encuentra las manzanas. Las dos CNN clasifican cada recorte por variedad y estado. ByteTrack mantiene los IDs.

Los tres modelos ya están guardados en Drive. Ejecutá este notebook de arriba hacia abajo con GPU; no hace falta volver a entrenar.

In [1]:
%pip install -q ultralytics==8.4.192 lap==0.5.12

In [2]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import time
import cv2
import numpy as np
import torch
from torch import nn
from torchvision import transforms
from PIL import Image
from ultralytics import YOLO
from IPython.display import display, Video
from google.colab import drive

drive.mount('/content/drive')
carpeta = Path('/content/drive/MyDrive/Inspeccion de manzanas - Colab')
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', dispositivo)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cuda


In [3]:
def cargar_cnn(archivo):
    assert archivo.is_file(), f'Falta {archivo.name}: ejecutá primero su notebook de entrenamiento.'
    datos = torch.load(archivo, map_location='cpu', weights_only=True)
    red = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
        nn.Linear(1024, 64), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(64, len(datos['clases'])),
    )
    red.load_state_dict(datos['pesos'])
    red.to(dispositivo).eval()
    preparar = transforms.Compose([
        transforms.Resize((datos['tamano'], datos['tamano'])),
        transforms.ToTensor(),
        transforms.Normalize(datos['media'], datos['desvio']),
    ])
    return red, datos['clases'], preparar

cnn_variedad, variedades, preparar_variedad = cargar_cnn(carpeta / 'cnn_variedades.pt')
cnn_estado, estados, preparar_estado = cargar_cnn(carpeta / 'cnn_estado.pt')
yolo = YOLO(carpeta / 'yolo_manzanas_best.pt')
print('Variedades:', variedades)
print('Estados:', estados)
print('Detector:', yolo.names)

Variedades: ['Crimson Snow', 'Granny Smith', 'Pink Lady', 'Red Delicious']
Estados: ['Defectuosa', 'No defectuosa']
Detector: {0: 'manzana'}


## Procesar una imagen
Las dos CNN reciben los mismos recortes, cada una con su preparación. Agrupamos las manzanas en un lote para evitar una llamada por manzana. El porcentaje es el score del modelo, no una garantía de acierto.

In [4]:
umbral = 0.75  # Umbral de demostración; todavía no está calibrado.

@torch.inference_mode()
def procesar_imagen(imagen):
    deteccion = yolo.track(imagen, persist=True, tracker='bytetrack.yaml',
                           conf=0.35, imgsz=640, max_det=100, device=dispositivo,
                           verbose=False)[0]
    dibujada = imagen.copy()
    alto, ancho = imagen.shape[:2]
    cajas, recortes, ids = [], [], []
    ids_detectados = [-1] * len(deteccion.boxes)
    if deteccion.boxes.id is not None:
        ids_detectados = deteccion.boxes.id.cpu().tolist()
    for indice, caja in enumerate(deteccion.boxes.xyxy.cpu().numpy().astype(int)):
        x1, y1, x2, y2 = caja
        x1, x2 = np.clip([x1, x2], 0, ancho)
        y1, y2 = np.clip([y1, y2], 0, alto)
        if x2 <= x1 or y2 <= y1:
            continue
        rgb = cv2.cvtColor(imagen[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
        recortes.append(Image.fromarray(rgb))
        cajas.append((int(x1), int(y1), int(x2), int(y2)))
        ids.append(int(ids_detectados[indice]))
    if not recortes:
        return dibujada, []

    lote_variedad = torch.stack([preparar_variedad(foto) for foto in recortes]).to(dispositivo)
    lote_estado = torch.stack([preparar_estado(foto) for foto in recortes]).to(dispositivo)
    pred_variedad = cnn_variedad(lote_variedad).softmax(1).cpu()
    pred_estado = cnn_estado(lote_estado).softmax(1).cpu()
    filas = []
    for i, (x1, y1, x2, y2) in enumerate(cajas):
        confianza_v, clase_v = pred_variedad[i].max(0)
        confianza_e, clase_e = pred_estado[i].max(0)
        variedad, estado = variedades[int(clase_v)], estados[int(clase_e)]
        etiqueta_v = variedad if confianza_v >= umbral else 'Variedad por revisar'
        etiqueta_e = estado if confianza_e >= umbral else 'Estado por revisar'
        identificador = str(ids[i]) if ids[i] >= 0 else '-'
        color = (200, 120, 40)
        cv2.rectangle(dibujada, (x1, y1), (x2, y2), color, 2)
        lineas = [f'ID {identificador} | {etiqueta_v}', f'{etiqueta_e} ({confianza_e:.0%})']
        for numero, linea in enumerate(lineas):
            ancho_texto = cv2.getTextSize(linea, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)[0][0]
            x = max(0, min(x1, ancho - ancho_texto - 4))
            y = max(16, min(y1 + 16 + 17*numero, alto - 4))
            cv2.rectangle(dibujada, (x, y-14), (x+ancho_texto+4, y+3), (30, 30, 30), -1)
            cv2.putText(dibujada, linea, (x+2, y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
        filas.append([ids[i], x1, y1, x2, y2, variedad, float(confianza_v), estado, float(confianza_e)])
    return dibujada, filas

## Probar un video
Elegí uno de los videos de Drive. `segundos = 0` procesa el archivo completo. El resultado incluye video, predicciones CSV y tiempos medidos. No hay etiquetas reales de variedad o estado para estos videos: sirven como demo, no para calcular exactitud.

Se procesa a un ancho máximo de 960 píxeles. `vista_previa = True` permite ver el avance; dejalo en `False` para medir sin esa salida extra. Las coordenadas del CSV corresponden a la resolución procesada.

In [5]:
nombre_video = 'manzanas_en_cinta.mp4'
segundos = 10
ancho_maximo = 960
vista_previa = False
video = Path('/content') / nombre_video
shutil.copy2(carpeta / nombre_video, video)
captura = cv2.VideoCapture(str(video))
assert captura.isOpened(), 'No se pudo abrir el video.'
fps_video = captura.get(cv2.CAP_PROP_FPS)
assert fps_video > 0, 'El video no informa sus FPS.'
ancho = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))
escala = min(1, ancho_maximo / ancho)
ancho = int(ancho * escala) // 2 * 2
alto = int(alto * escala) // 2 * 2

salida = Path('/content/resultado_temporal.mp4')
escritor = cv2.VideoWriter(str(salida), cv2.VideoWriter_fourcc(*'mp4v'), fps_video, (ancho, alto))
assert escritor.isOpened(), 'No se pudo crear el archivo de salida.'

# Calentar los modelos y reiniciar el seguimiento antes de medir.
ok, imagen = captura.read()
assert ok, 'El video no tiene imágenes legibles.'
procesar_imagen(cv2.resize(imagen, (ancho, alto)))
for tracker in yolo.predictor.trackers:
    tracker.reset()
captura.set(cv2.CAP_PROP_POS_FRAMES, 0)

True

In [6]:
tiempos, filas = [], []
cuadro = 0
if vista_previa:
    vista_video = display('', display_id=True)
inicio = time.perf_counter()
try:
    while segundos == 0 or cuadro < int(segundos * fps_video):
        ok, imagen = captura.read()
        if not ok:
            break
        t0 = time.perf_counter()
        imagen = cv2.resize(imagen, (ancho, alto))
        dibujada, predicciones = procesar_imagen(imagen)
        tiempos.append(time.perf_counter() - t0)
        escritor.write(dibujada)
        for prediccion in predicciones:
            filas.append([cuadro, cuadro / fps_video] + prediccion)
        cuadro += 1
        if vista_previa and cuadro % 10 == 0:
            vista_video.update(Image.fromarray(cv2.cvtColor(dibujada, cv2.COLOR_BGR2RGB)))
finally:
    captura.release()
    escritor.release()
duracion = time.perf_counter() - inicio
assert cuadro > 0, 'No se procesaron fotogramas.'

medidas = {
    'video': nombre_video, 'dispositivo': dispositivo, 'cuadros': cuadro,
    'resolucion': [ancho, alto], 'vista_previa': vista_previa,
    'fps_video': fps_video, 'fps_modelos_y_dibujo': cuadro / sum(tiempos),
    'fps_lectura_procesamiento_escritura': cuadro / duracion,
    'latencia_media_ms': float(np.mean(tiempos) * 1000),
    'latencia_p95_ms': float(np.percentile(tiempos, 95) * 1000),
}
print(json.dumps(medidas, indent=2))
print('Medición sin transcodificación, transferencia a Drive ni reproducción en el navegador.')

{
  "video": "manzanas_en_cinta.mp4",
  "dispositivo": "cuda",
  "cuadros": 231,
  "resolucion": [
    960,
    540
  ],
  "vista_previa": false,
  "fps_video": 29.97002997002997,
  "fps_modelos_y_dibujo": 8.943278537090583,
  "fps_lectura_procesamiento_escritura": 7.316728702991657,
  "latencia_media_ms": 111.81581741558045,
  "latencia_p95_ms": 235.5886495000732
}
Medición sin transcodificación, transferencia a Drive ni reproducción en el navegador.


In [7]:
nombre = video.stem + '_resultado'
web = Path('/content') / (nombre + '.mp4')
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(salida),
                '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(web)], check=True)
shutil.copy2(web, carpeta / web.name)
with open(carpeta / (nombre + '.csv'), 'w') as archivo:
    tabla = csv.writer(archivo)
    tabla.writerow(['cuadro','segundo','id','x1','y1','x2','y2','variedad','score_variedad','estado','score_estado'])
    tabla.writerows(filas)
with open(carpeta / (nombre + '_tiempos.json'), 'w') as archivo:
    json.dump(medidas, archivo, indent=2)
display(Video(str(web), embed=True, width=800))

Output hidden; open in https://colab.research.google.com to view.

## Cámara en vivo (opcional)
Cambiá `usar_camara` a `True` y ejecutá esta celda. Pedirá permiso de cámara; no usa micrófono. Envía una imagen por vez al Colab y muestra la respuesta, sin acumular imágenes en espera. La conexión puede limitar los FPS. Detené con el botón o interrumpí la celda.

In [8]:
usar_camara = False

if usar_camara:
    import base64
    from google.colab.output import eval_js
    from IPython.display import Javascript

    display(Javascript("""
    window.camaraManzanas = {activa: true};
    window.iniciarManzanas = async function() {
      const c = window.camaraManzanas;
      c.video = document.createElement('video');
      c.video.width = 640;
      c.boton = document.createElement('button');
      c.boton.textContent = 'Detener cámara';
      document.body.append(c.boton, c.video);
      c.stream = await navigator.mediaDevices.getUserMedia({video: {width: 640}, audio: false});
      c.video.srcObject = c.stream;
      await c.video.play();
      c.boton.onclick = () => {
        c.activa = false;
        c.stream.getTracks().forEach(t => t.stop());
      };
    };
    window.leerManzanas = function() {
      const c = window.camaraManzanas;
      if (!c.activa) return null;
      const canvas = document.createElement('canvas');
      canvas.width = c.video.videoWidth;
      canvas.height = c.video.videoHeight;
      canvas.getContext('2d').drawImage(c.video, 0, 0);
      return canvas.toDataURL('image/jpeg', 0.8);
    };
    window.cerrarManzanas = function() {
      const c = window.camaraManzanas;
      if (c.stream) c.stream.getTracks().forEach(t => t.stop());
      c.video?.remove(); c.boton?.remove(); c.activa = false;
    };
    """))
    vista = display('', display_id=True)
    if yolo.predictor is not None and hasattr(yolo.predictor, 'trackers'):
        for tracker in yolo.predictor.trackers:
            tracker.reset()
    try:
        eval_js('iniciarManzanas()')
        while True:
            t0 = time.perf_counter()
            foto = eval_js('leerManzanas()')
            if foto is None:
                break
            contenido = np.frombuffer(base64.b64decode(foto.split(',')[1]), np.uint8)
            imagen = cv2.imdecode(contenido, cv2.IMREAD_COLOR)
            dibujada, _ = procesar_imagen(imagen)
            fps = 1 / (time.perf_counter() - t0)
            cv2.putText(dibujada, f'{fps:.1f} FPS captura + inferencia', (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            vista.update(Image.fromarray(cv2.cvtColor(dibujada, cv2.COLOR_BGR2RGB)))
    finally:
        eval_js('cerrarManzanas()')
else:
    print('Cámara desactivada. El video se puede probar sin encenderla.')

Cámara desactivada. El video se puede probar sin encenderla.


El modelo de estado busca defectos visibles; no confirma sanidad interna. La variedad está limitada a las cuatro clases entrenadas. Un score alto en escenas nuevas puede ser incorrecto.

[YOLO y ByteTrack](https://docs.ultralytics.com/modes/track/) · Los pesos YOLO son los del grupo (`pruebas/yolo_manzanas_best.pt`, remoto `4a89acc`).

Pruebas del 17/11/2026 en una T4: **28,1 FPS** en `cinta_industrial` y **9,2 FPS** en `manzanas_en_cinta`, incluyendo lectura y escritura. La segunda escena tiene más detecciones por cuadro. No alcanza 30 FPS en ambas.

La integración funciona, pero la clasificación en estas escenas todavía no es fiable: predice «defectuosa» en casi todos los recortes y el detector omite muchas manzanas pequeñas. Los scores no son probabilidades calibradas. Necesitamos recortes de escenas reales con variedad y estado conocidos para mejorar y evaluar esa parte. La cámara está implementada, pero todavía no se probó con una webcam física.

La ejecución completa desde una sesión reiniciada terminó sin errores. En esa repetición, `manzanas_en_cinta` dio **7,3 FPS**: el rendimiento de Colab puede variar entre ejecuciones.